In [ ]:
%pip install openai

# Deepseek API 调用

In [1]:
# from openai import OpenAI

# client = OpenAI(
#     api_key="sk-7ff1bb81362f4d4a83bd00f0a1d5b822",
#     base_url="https://api.deepseek.com"
# )

# response = client.chat.completions.create(
#     model="deepseek-chat",
#     messages=[
#         {"role": "system", "content": "You are a helpful assistant."},
#         {"role": "user", "content": "Hello!"}
#     ],
#     stream=False
# )

# print(response.choices[0].message.content)

# 流式输出
# stream = client.chat.completions.create(
#     model="deepseek-chat",
#     messages=[{"role": "user", "content": "Hello!"}],
#     stream=True
# )

# for chunk in stream:
#     if chunk.choices[0].delta.content:
#           print(chunk.choices[0].delta.content, end="")
# -------------------------------------------------------------------------------------------------------------------

# 上面是deepseek的基础调用
import os
from openai import OpenAI
# from dotenv import load_dotenv
from typing import List, Dict

# 加载 .env 文件中的环境变量
# load_dotenv()

api_key="sk-7ff1bb81362f4d4a83bd00f0a1d5b822"
base_url="https://api.deepseek.com"
model="deepseek-chat"

class DeepseekLLM:
    """
    为本书 "Hello Agents" 定制的LLM客户端。
    它用于调用任何兼容OpenAI接口的服务，并默认使用流式响应。
    """
    def __init__(self, model: str = None, apiKey: str = None, baseUrl: str = None, timeout: int = None):
        """
        初始化客户端。优先使用传入参数，如果未提供，则从环境变量加载。
        """
        self.model = model or os.getenv("LLM_MODEL_ID")
        apiKey = apiKey or os.getenv("LLM_API_KEY")
        baseUrl = baseUrl or os.getenv("LLM_BASE_URL")
        timeout = timeout or int(os.getenv("LLM_TIMEOUT", 60))
        
        if not all([self.model, apiKey, baseUrl]):
            raise ValueError("模型ID、API密钥和服务地址必须被提供或在.env文件中定义。")

        self.client = OpenAI(api_key=apiKey, base_url=baseUrl, timeout=timeout)

    def think(self, messages: List[Dict[str, str]], temperature: float = 0) -> str:
        """
        调用大语言模型进行思考，并返回其响应。
        """
        print(f"🧠 正在调用 {self.model} 模型...")
        try:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=messages,
                temperature=temperature,
                stream=True,
            )
            
            # 处理流式响应
            print("✅ 大语言模型响应成功:")
            collected_content = []
            for chunk in response:
                if not chunk.choices:
                    continue
                content = chunk.choices[0].delta.content or ""
                print(content, end="", flush=True)
                collected_content.append(content)
            print()  # 在流式输出结束后换行
            return "".join(collected_content)

        except Exception as e:
            print(f"❌ 调用LLM API时发生错误: {e}")
            return None

# --- 客户端使用示例 ---
if __name__ == '__main__':
    try:
        # 进行显示的初始化
        llmClient = DeepseekLLM(model, api_key, base_url)
        exampleMessages = [
            {"role": "system", "content": "You are a helpful assistant that writes Python code."},
            {"role": "user", "content": "写一个快速排序算法"}
        ]
        
        print("--- 调用LLM ---")
        responseText = llmClient.think(exampleMessages)
        if responseText:
            print("\n\n--- 完整模型响应 ---")
            print(responseText)

    except ValueError as e:
        print(e)

--- 调用LLM ---
🧠 正在调用 deepseek-chat 模型...
✅ 大语言模型响应成功:
我来为你写一个快速排序算法的Python实现：

```python
def quick_sort(arr):
    """
    快速排序算法实现
    
    参数:
        arr: 待排序的列表
    
    返回:
        排序后的列表
    """
    # 基本情况：如果列表为空或只有一个元素，直接返回
    if len(arr) <= 1:
        return arr
    
    # 选择基准元素（这里选择中间元素）
    pivot = arr[len(arr) // 2]
    
    # 将列表分为三部分：小于基准、等于基准、大于基准
    left = [x for x in arr if x < pivot]
    middle = [x for x in arr if x == pivot]
    right = [x for x in arr if x > pivot]
    
    # 递归排序左右两部分，然后合并结果
    return quick_sort(left) + middle + quick_sort(right)


# 原地排序版本（更高效，节省内存）
def quick_sort_inplace(arr, low=0, high=None):
    """
    原地快速排序算法实现
    
    参数:
        arr: 待排序的列表
        low: 起始索引
        high: 结束索引
    """
    if high is None:
        high = len(arr) - 1
    
    if low < high:
        # 分区操作，返回基准元素的最终位置
        pivot_index = partition(arr, low, high)
        
        # 递归排序基准元素左右两边的子数组
        quick_sort_inplace(arr, low, pivot_index - 1)
        quick_so

# 1. ReAct
ReAct的巧妙之处在于，它认识到思考与行动是相辅相成的。思考指导行动，而行动的结果又反过来修正思考。智能体将不断重复这个 Thought -> Action -> Observation 的循环，将新的观察结果追加到历史记录中，形成一个不断增长的上下文，直到它在Thought中认为已经找到了最终答案，然后输出结果。这个过程形成了一个强大的协同效应：推理使得行动更具目的性，而行动则为推理提供了事实依据。

![image.png](figures\image-0.png)

这种机制特别适用于以下场景：
需要外部知识的任务：如查询实时信息（天气、新闻、股价）、搜索专业领域的知识等。
需要精确计算的任务：将数学问题交给计算器工具，避免LLM的计算错误。
需要与API交互的任务：如操作数据库、调用某个服务的API来完成特定功能。

### 1.1.外部工具 (Tools)
如果说大语言模型是智能体的大脑，那么工具 (Tools) 就是其与外部世界交互的“手和脚”。为了让ReAct范式能够真正解决我们设定的问题，智能体需要具备调用外部工具的能力。
针对这个问题，我们调用 Image-2 来绘制一张图片，作为外部的工具
一个良好定义的工具应包含以下三个核心要素：

1. 名称 (Name)： 一个简洁、唯一的标识符，供智能体在 Action 中调用，例如 Search。
2. 描述 (Description)： 一段清晰的自然语言描述，说明这个工具的用途。这是整个机制中最关键的部分，因为大语言模型会依赖这段描述来判断何时使用哪个工具。
3. 执行逻辑 (Execution Logic)： 真正执行任务的函数或方法。

In [11]:
from openai import OpenAI
import base64
from PIL import Image

def tool_Image2generator(ref_path, out_path,prompt):
    client = OpenAI(
        api_key="sk-fJifEDO3ScSHH9U1xtXKZPKeuu8t2yrRBTMBBEWrKEf4aQGC",
        base_url="https://api.aipaibox.com/v1",
    )

    # 修正2：手动打开文件，并保证在 API 调用期间保持打开
    with open(ref_path, "rb") as img_file:
        print("开始生成图片~")
        res = client.images.edit(
            model="gpt-image-2",
            image=img_file,          # 直接传文件对象，不要包在列表里
            prompt= prompt,
            size="3840x2160",
        )

    # 保存结果
    with open(out_path, "wb") as f:
        f.write(base64.b64decode(res.data[0].b64_json))
    
    return f"图片已经生成，位置在:{out_path}"

# 提示词
prompt = """把上传照片转化成一种可爱混乱的儿童蜡笔涂鸦插画风格。
            整体像是彩铅 + 蜡笔 + 手账贴纸 + MS Paint 涂鸦的结合。
            使用：粗糙线条、抖动边缘、不均匀上色、纸张纹理、彩铅叠色、蜡笔颗粒感、故意幼稚但很有灵气的画风。人物变成Q版萌系二次元风格，大眼睛，夸张表情，可爱活泼。
            背景加入大量 doodle：
            爱心、星星、糖果、笑脸云朵、小花、贴纸、游戏UI元素、乱涂乱画符号。
            颜色以：粉色、蓝色、紫色、奶黄色、薄荷色为主。
            整体氛围：可爱、混乱、梦幻、少女感、游戏宅手账风、互联网 kawaii，Kawaii aesthetic 不要精致，不要高级商业插画感，不要真实渲染，不要干净线稿，保留“手绘失败感”和“乱涂鸦感”。
            """

# 修正1：使用原始字符串处理路径
ref_path = r"figures\image-gpt\image-ref.png"
out_path = r"figures\image-gpt\image-out.png"

# 执行工具
tool_Image2generator(ref_path, out_path, prompt)

# 图片展示
img = Image.open(out_path)
img   # 打开系统默认图片查看器

开始生成图片~


InternalServerError: Error code: 524 - {'error': {'message': 'The origin web server did not return a complete response within the 120-second Proxy Read Timeout window. The connection was established, but the origin took too long to respond.', 'type': 'bad_response_status_code', 'param': '', 'code': 'bad_response_status_code'}}

### 1.2 工具执行器
构建通用的工具执行器
当智能体需要使用多种工具时（例如，除了搜索，还可能需要计算、查询数据库等），我们需要一个统一的管理器来注册和调度这些工具。为此，我们创建一个 ToolExecutor 类。

In [3]:
from typing import Dict, Any

class ToolExecutor:
    """
    一个工具执行器，负责管理和执行工具。
    """
    def __init__(self):
        self.tools: Dict[str, Dict[str, Any]] = {}

    def registerTool(self, name: str, description: str, func: callable):
        """
        向工具箱中注册一个新工具。
        """
        if name in self.tools:
            print(f"警告:工具 '{name}' 已存在，将被覆盖。")
        self.tools[name] = {"description": description, "func": func}
        print(f"工具 '{name}' 已注册。")

    def getTool(self, name: str) -> callable:
        """
        根据名称获取一个工具的执行函数。
        """
        return self.tools.get(name, {}).get("func")

    def getAvailableTools(self) -> str:
        """
        获取所有可用工具的格式化描述字符串。
        """
        return "\n".join([
            f"- {name}: {info['description']}" 
            for name, info in self.tools.items()
        ])
# --- 工具初始化与使用示例 ---
# if __name__ == '__main__':
#     # 1. 初始化工具执行器
#     toolExecutor = ToolExecutor()

#     # 2. 注册我们的实战搜索工具
#     # search_description = "一个网页搜索引擎。当你需要回答关于时事、事实以及在你的知识库中找不到的信息时，应使用此工具。"
#     Painting_description = "一个风格转化的绘画工具，当你需要进行绘画风格转化的时候，使用此工具"
#     toolExecutor.registerTool("Painting", Painting_description, tool_Image2generator)
    
#     # 3. 打印可用的工具
#     print("\n--- 可用的工具 ---")
#     print(toolExecutor.getAvailableTools())

#     # 4. 智能体的Action调用，这次我们问一个实时性的问题
#     print("\n--- 执行 绘画: 绘图['风格转化~'] ---")
#     tool_name = "Painting"
#     tool_input = r"figures\image-gpt\image-ref.png"
#     tool_output = r"figures\image-gpt\image-ref.png"

#     tool_function = toolExecutor.getTool(tool_name)
#     if tool_function:
#         observation = tool_function(tool_input,tool_output)
#         print("--- 结果 (Observation) ---")
#         print(observation)
#     else:
#         print(f"错误:未找到名为 '{tool_name}' 的工具。")

### 1.3 ReAct 智能体 编码实现
1. 系统提示词设计
这个模板定义了智能体与LLM之间交互的规范：

1.1. 角色定义： “你是一个有能力调用外部工具的智能助手”，设定了LLM的角色。

1.2. 工具清单 ({tools})： 告知LLM它有哪些可用的“手脚”。

1.3. 格式规约 (Thought/Action)： 这是最重要的部分，它强制LLM的输出具有结构性，使我们能通过代码精确解析其意图。

1.4. 动态上下文 ({question}/{history})： 将用户的原始问题和不断累积的交互历史注入，让LLM基于完整的上下文进行决策。

2. 核心循环的实现

ReActAgent 的核心是一个循环，它不断地“格式化提示词 -> 调用LLM -> 执行动作 -> 整合结果”，直到任务完成或达到最大步数限制。

3. 输出解析器的实现
LLM 返回的是纯文本，我们需要从中精确地提取出Thought和Action。这是通过几个辅助解析函数完成的，它们通常使用正则表达式来实现。

4. 工具调用与执行
这段代码是Action的执行中心。它首先检查是否为Finish指令，如果是，则流程结束。否则，它会通过tool_executor获取对应的工具函数并执行，得到observation。

5. 观测结果的整合
最后一步，也是形成闭环的关键，是将Action本身和工具执行后的Observation添加回历史记录中，为下一轮循环提供新的上下文。

6. 运行实例与分析

In [5]:
import re
# from llm_client import HelloAgentsLLM
# from tools import ToolExecutor, search
# tool_Image2generator; ToolExecutor

# 提示词
REACT_PROMPT_TEMPLATE = """
请注意，你是一个有能力调用外部工具的智能助手。

可用工具如下：
{tools}

请严格按照以下格式进行回应：

Thought: 你的思考过程，用于分析问题、拆解任务和规划下一步行动。
Action: 你决定采取的行动，必须是以下格式之一：
- `{{tool_name}}[{{tool_input}}]`：调用一个可用工具。
- `Finish[完成功能]`：当你认为已经获得完成相关功能时。
- 当你执行完成相关功能的时候，能够给用户一个功能完成的反馈时，你必须在`Action:`字段后使用 `Finish[最终答案]` 来输出最终反馈。

现在，请开始:
按照风格提示{description};开始绘画
"""
class ReActAgent:
    def __init__(self, llm_client: DeepseekLLM, tool_executor: ToolExecutor, max_steps: int = 2):
        self.llm_client = llm_client
        self.tool_executor = tool_executor
        self.max_steps = max_steps
        self.history = []

    def run(self, description: str, tool_input_path: str, tool_output_path: str):
        self.history = []
        current_step = 0

        while current_step < self.max_steps:
            current_step += 1
            print(f"\n--- 第 {current_step} 步 ---")

            tools_desc = self.tool_executor.getAvailableTools()
            history_str = "\n".join(self.history)
            prompt = REACT_PROMPT_TEMPLATE.format(tools=tools_desc, description=description, history=history_str)
        
            messages = [{"role": "user", "content": prompt}]
            response_text = self.llm_client.think(messages=messages)
            if not response_text:
                print("错误：LLM未能返回有效响应。"); break

            thought, action = self._parse_output(response_text)
            if thought: print(f"🤔 思考: {thought}")
            if not action: print("警告：未能解析出有效的Action，流程终止。"); break
            
            if action.startswith("Finish"):
                # 如果是Finish指令，提取最终答案并结束
                final_answer = self._parse_action_input(action)
                print(f"🎉 最终答案: {final_answer}")
                return final_answer
            
            tool_name, tool_input = self._parse_action(action)
            if not tool_name or not tool_input:
                self.history.append("Observation: 无效的Action格式，请检查。"); continue

            print(f"🎬 行动: {tool_name}[{tool_input}]")
            tool_function = self.tool_executor.getTool(tool_name)

            observation = tool_function(tool_input_path, tool_output_path, description) if tool_function else f"错误：未找到名为 '{tool_name}' 的工具。"
            
            print(f"👀 观察: {observation}")
            self.history.append(f"Action: {action}")
            self.history.append(f"Observation: {observation}")

        print("已达到最大步数，流程终止。")
        return None

    def _parse_output(self, text: str):
        # Thought: 匹配到 Action: 或文本末尾
        thought_match = re.search(r"Thought:\s*(.*?)(?=\nAction:|$)", text, re.DOTALL)
        # Action: 匹配到文本末尾
        action_match = re.search(r"Action:\s*(.*?)$", text, re.DOTALL)
        thought = thought_match.group(1).strip() if thought_match else None
        action = action_match.group(1).strip() if action_match else None
        return thought, action

    def _parse_action(self, action_text: str):
        match = re.match(r"(\w+)\[(.*)\]", action_text, re.DOTALL)
        return (match.group(1), match.group(2)) if match else (None, None)

    def _parse_action_input(self, action_text: str):
        match = re.match(r"\w+\[(.*)\]", action_text, re.DOTALL)
        return match.group(1) if match else ""

if __name__ == '__main__':
    api_key="sk-7ff1bb81362f4d4a83bd00f0a1d5b822"
    base_url="https://api.deepseek.com"
    model="deepseek-chat"
    llm = DeepseekLLM(model, api_key, base_url)
    tool_executor = ToolExecutor()
    # search_desc = "一个网页搜索引擎。当你需要回答关于时事、事实以及在你的知识库中找不到的信息时，应使用此工具。"
    Painting_description = "一个风格转化的绘画工具，当你需要进行绘画风格转化的时候，使用此工具"
    tool_executor.registerTool("Painting", Painting_description, tool_Image2generator)
    agent = ReActAgent(llm_client=llm, tool_executor=tool_executor)
    #question = "华为最新的手机是哪一款？它的主要卖点是什么？"

    prompt="""把上传照片转化成一种可爱混乱的儿童蜡笔涂鸦插画风格。
                整体像是彩铅 + 蜡笔 + 手账贴纸 + MS Paint 涂鸦的结合。
                使用：粗糙线条、抖动边缘、不均匀上色、纸张纹理、彩铅叠色、蜡笔颗粒感、故意幼稚但很有灵气的画风。人物变成Q版萌系二次元风格，大眼睛，夸张表情，可爱活泼。
                背景加入大量 doodle：
                爱心、星星、糖果、笑脸云朵、小花、贴纸、游戏UI元素、乱涂乱画符号。
                颜色以：粉色、蓝色、紫色、奶黄色、薄荷色为主。
                整体氛围：可爱、混乱、梦幻、少女感、游戏宅手账风、互联网 kawaii，Kawaii aesthetic 不要精致，不要高级商业插画感，不要真实渲染，不要干净线稿，保留“手绘失败感”和“乱涂鸦感”。
                """
    
    tool_input_path = r"figures\image-gpt\image-ref.png"
    tool_output_path = r"figures\image-gpt\image-ref.png"
    agent.run(prompt,tool_input_path,tool_output_path)

工具 'Painting' 已注册。

--- 第 1 步 ---
🧠 正在调用 deepseek-chat 模型...
✅ 大语言模型响应成功:
Thought: 用户希望将上传的照片转化为一种特定风格的儿童蜡笔涂鸦插画，风格描述非常详细，包括线条、纹理、颜色、背景元素和整体氛围。我需要调用绘画工具来生成这张图片。工具是Painting，用于风格转化。我将把用户的详细风格描述作为输入传递给工具。

Action: Painting[按照风格提示把上传照片转化成一种可爱混乱的儿童蜡笔涂鸦插画风格。
整体像是彩铅 + 蜡笔 + 手账贴纸 + MS Paint 涂鸦的结合。
使用：粗糙线条、抖动边缘、不均匀上色、纸张纹理、彩铅叠色、蜡笔颗粒感、故意幼稚但很有灵气的画风。人物变成Q版萌系二次元风格，大眼睛，夸张表情，可爱活泼。
背景加入大量 doodle：
爱心、星星、糖果、笑脸云朵、小花、贴纸、游戏UI元素、乱涂乱画符号。
颜色以：粉色、蓝色、紫色、奶黄色、薄荷色为主。
整体氛围：可爱、混乱、梦幻、少女感、游戏宅手账风、互联网 kawaii，Kawaii aesthetic 不要精致，不要高级商业插画感，不要真实渲染，不要干净线稿，保留“手绘失败感”和“乱涂鸦感”。
;开始绘画]
🤔 思考: 用户希望将上传的照片转化为一种特定风格的儿童蜡笔涂鸦插画，风格描述非常详细，包括线条、纹理、颜色、背景元素和整体氛围。我需要调用绘画工具来生成这张图片。工具是Painting，用于风格转化。我将把用户的详细风格描述作为输入传递给工具。
🎬 行动: Painting[按照风格提示把上传照片转化成一种可爱混乱的儿童蜡笔涂鸦插画风格。
整体像是彩铅 + 蜡笔 + 手账贴纸 + MS Paint 涂鸦的结合。
使用：粗糙线条、抖动边缘、不均匀上色、纸张纹理、彩铅叠色、蜡笔颗粒感、故意幼稚但很有灵气的画风。人物变成Q版萌系二次元风格，大眼睛，夸张表情，可爱活泼。
背景加入大量 doodle：
爱心、星星、糖果、笑脸云朵、小花、贴纸、游戏UI元素、乱涂乱画符号。
颜色以：粉色、蓝色、紫色、奶黄色、薄荷色为主。
整体氛围：可爱、混乱、梦幻、少女感、游戏宅手账风、互联网 kawaii，Kawaii aesthetic 不要精致，不要高级商业插画感，不要真实渲染，不要干净线稿，保留“手绘失败感”和“

InternalServerError: Error code: 503 - {'error': {'message': 'auth_unavailable: no auth available (providers=codex, model=gpt-image-2)', 'type': 'server_error', 'param': '', 'code': 'internal_server_error'}}

### 1.4 总结
智能体将不断重复这个 Thought -> Action -> Observation 的循环，将新的观察结果追加到历史记录中，形成一个不断增长的上下文，直到它在Thought中认为已经找到了最终答案，然后输出结果。这个过程形成了一个强大的协同效应：推理使得行动更具目的性，而行动则为推理提供了事实依据。

1）ReAct 的主要特点

高可解释性：ReAct 最大的优点之一就是透明。通过 Thought 链，我们可以清晰地看到智能体每一步的“心路历程”——它为什么会选择这个工具，下一步又打算做什么。这对于理解、信任和调试智能体的行为至关重要。

动态规划与纠错能力：与一次性生成完整计划的范式不同，ReAct 是“走一步，看一步”。它根据每一步从外部世界获得的 Observation 来动态调整后续的 Thought 和 Action。如果上一步的搜索结果不理想，它可以在下一步中修正搜索词，重新尝试。

工具协同能力：ReAct 范式天然地将大语言模型的推理能力与外部工具的执行能力结合起来。LLM 负责运筹帷幄（规划和推理），工具负责解决具体问题（搜索、计算），二者协同工作，突破了单一 LLM 在知识时效性、计算准确性等方面的固有局限。

（2）ReAct 的固有局限性

对LLM自身能力的强依赖：ReAct 流程的成功与否，高度依赖于底层 LLM 的综合能力。如果 LLM 的逻辑推理能力、指令遵循能力或格式化输出能力不足，就很容易在 Thought 环节产生错误的规划，或者在 Action 环节生成不符合格式的指令，导致整个流程中断。

执行效率问题：由于其循序渐进的特性，完成一个任务通常需要多次调用 LLM。每一次调用都伴随着网络延迟和计算成本。对于需要很多步骤的复杂任务，这种串行的“思考-行动”循环可能会导致较高的总耗时和费用。

提示词的脆弱性：整个机制的稳定运行建立在一个精心设计的提示词模板之上。模板中的任何微小变动，甚至是用词的差异，都可能影响 LLM 的行为。此外，并非所有模型都能持续稳定地遵循预设的格式，这增加了在实际应用中的不确定性。

可能陷入局部最优：步进式的决策模式意味着智能体缺乏一个全局的、长远的规划。它可能会因为眼前的 Observation 而选择一个看似正确但长远来看并非最优的路径，甚至在某些情况下陷入“原地打转”的循环中。

（3）调试技巧

当你构建的 ReAct 智能体行为不符合预期时，可以从以下几个方面入手进行调试：

检查完整的提示词：在每次调用 LLM 之前，将最终格式化好的、包含所有历史记录的完整提示词打印出来。这是追溯 LLM 决策源头的最直接方式。

分析原始输出：当输出解析失败时（例如，正则表达式没有匹配到 Action），务必将 LLM 返回的原始、未经处理的文本打印出来。这能帮助你判断是 LLM 没有遵循格式，还是你的解析逻辑有误。

验证工具的输入与输出：检查智能体生成的 tool_input 是否是工具函数所期望的格式，同时也要确保工具返回的 observation 格式是智能体可以理解和处理的。
调整提示词中的示例 (Few-shot Prompting)：如果模型频繁出错，可以在提示词中加入一两个完整的“Thought-Action-Observation”成功案例，通过示例来引导模型更好地遵循你的指令。

尝试不同的模型或参数：更换一个能力更强的模型，或者调整 temperature 参数（通常设为0以保证输出的确定性），有时能直接解决问题。

（4） 我的总结
感觉上这个agent 根据大模型的回答，返回需要调用的agent的相关信息，然后执行命令，观测历史，再次做决策